<a href="https://colab.research.google.com/github/e-eki/rsna-pneumonia-yolo-pipeline/blob/development/rsna_pneumonia_yolo_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# notebooks/colab_train.ipynb

In [ ]:
# ============================================================
# Ячейка 1: Монтирование Google Drive
# ============================================================
# Drive нужен только для сохранения артефактов (модели, runs, отчёты).
# Основная работа идёт в /content — там быстрее I/O.
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ============================================================
# Ячейка 2: Проверка GPU
# ============================================================
# Runtime → Change runtime type → T4 GPU
!nvidia-smi

In [ ]:
# ============================================================
# Ячейка 3: Клонирование репозитория
# ============================================================
!git clone https://github.com/e-eki/rsna-pneumonia-yolo-pipeline.git
%cd rsna-pneumonia-yolo-pipeline

In [ ]:
# ============================================================
# Ячейка 4: Установка зависимостей
# ============================================================
!pip install -q -r requirements.txt

In [ ]:
# ============================================================
# Ячейка 5: Настройка Kaggle API через Colab Secrets
# ============================================================
# Токен хранится в Colab Secrets (🔑 на левой панели), НЕ в коде.
# Name секрета: KAGGLE_API_TOKEN, Value: KGAT_...
import os

try:
    from google.colab import userdata
    os.environ["KAGGLE_API_TOKEN"] = userdata.get("KAGGLE_API_TOKEN")
    print("✅ KAGGLE_API_TOKEN загружен из Colab Secrets")
except Exception as e:
    raise RuntimeError(
        "Не удалось получить KAGGLE_API_TOKEN. "
        "Добавьте секрет с этим именем в 🔑 на левой панели Colab."
    ) from e

# Проверка, что CLI видит токен
!kaggle datasets list -s pneumonia | head -5

In [ ]:
# ============================================================
# Ячейка 6: Скачивание датасета
# ============================================================
!python src/download_data.py --config configs/config.yaml

# ============================================================
# ЭТАП 1: RAW — визуальная проверка исходников
# ============================================================
!python src/analyze_annotations.py --stage raw
!python src/sample_and_draw.py --stage raw --num 30 --grid
# Смотрим:
#   data/reports/raw/*.png              — статистика разметки
#   data/samples/raw/contact_sheet.jpg  — контактный лист


In [ ]:
# ============================================================
# Ячейка 7: Предобработка + фильтрация «плохой» разметки
# ============================================================
!python src/preprocess.py --config configs/config.yaml
# В логе — сводка: сколько боксов оставлено/отвергнуто,
# сколько изображений перемещено в staged/rejected_images/

# ============================================================
# ЭТАП 2: STAGED — проверяем, что данные стали чистыми
# ============================================================
!python src/analyze_annotations.py --stage staged
!python src/sample_and_draw.py --stage staged --num 50 --grid
# Смотрим:
#   data/reports/staged/*.png           — out_of_bounds должно быть 0
#   data/samples/staged/contact_sheet.jpg

# --- Смотрим, что именно отвергли ---
!python src/collect_rejected.py
# Создаётся data/samples/rejected/ с отвергнутыми снимками и красными боксами
# Локально смотреть: python src/viewer.py --dir data/samples/rejected

In [ ]:
# ============================================================
# Ячейка 8: Разделение на train/val и YOLO-разметка
# ============================================================
!python src/split_data.py --config configs/config.yaml

# ============================================================
# ЭТАП 3: PROCESSED — финальный датасет для модели
# ============================================================
!python src/analyze_annotations.py --stage processed --split train
!python src/analyze_annotations.py --stage processed --split val
!python src/sample_and_draw.py --stage processed --split train --num 50 --grid
!python src/sample_and_draw.py --stage processed --split val   --num 50 --grid

# ============================================================
# ГЛАВНАЯ проверка: RAW vs PROCESSED
# ============================================================
# Красный = raw (пиксельные координаты), зелёный = processed (YOLO).
# Прямоугольники должны точно совпадать.
!python src/compare_stages.py --num 40 --mode overlay --split train
!python src/compare_stages.py --num 40 --mode side    --split val
# Смотреть: data/samples/compare/

In [ ]:
# ============================================================
# Ячейка 9: Обучение YOLO
# ============================================================
!python src/train.py --config configs/config.yaml

In [ ]:
# ============================================================
# Ячейка 10: Оценка на валидации
# ============================================================
!python src/evaluate.py --config configs/config.yaml

In [ ]:
# ============================================================
# Ячейка 11: Сохранение результатов на Google Drive
# ============================================================
import shutil
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive/rsna-pneumonia")
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)

# Модели и логи обучения
for src_dir, dst_name in [
    ("runs", "runs"),
    ("models", "models"),
    ("data/reports", "reports"),
    ("data/samples", "samples"),
]:
    src = Path(src_dir)
    if src.exists():
        shutil.copytree(src, DRIVE_ROOT / dst_name, dirs_exist_ok=True)
        print(f"✅ Сохранено: {src_dir} → {DRIVE_ROOT / dst_name}")
    else:
        print(f"⚠️ Пропущено (нет папки): {src_dir}")

print(f"\nВсе результаты на Google Drive: {DRIVE_ROOT}")


In [ ]:
# ============================================================
# Ячейка 12 (опционально): Итоговая сводка по пайплайну
# ============================================================
import json
from pathlib import Path
import pandas as pd

print("=" * 60)
print("СВОДКА ПО ПАЙПЛАЙНУ")
print("=" * 60)

# Отчёты по стадиям
for stage_dir in ["data/reports/raw", "data/reports/staged",
                  "data/reports/processed_train", "data/reports/processed_val"]:
    report = Path(stage_dir) / "report.json"
    if report.exists():
        data = json.load(open(report))
        print(f"\n[{stage_dir}]")
        print(f"  боксов:               {data.get('num_boxes')}")
        bpi = data.get("boxes_per_image", {})
        print(f"  изображений с боксами: {bpi.get('with_boxes')}")
        print(f"  изображений без боксов:{bpi.get('without_boxes')}")
        if "yolo_out_of_bounds" in data:
            print(f"  YOLO out-of-bounds:    {data['yolo_out_of_bounds']}")
        if "pixel_out_of_bounds_images" in data:
            print(f"  pixel out-of-bounds:   {data['pixel_out_of_bounds_images']}")

# Отчёт фильтрации
filt = Path("data/staged/filtering_report.json")
if filt.exists():
    data = json.load(open(filt))
    print(f"\n[Фильтрация]")
    print(f"  входных боксов:       {data.get('total_input_boxes')}")
    print(f"  оставлено:            {data.get('kept_boxes')}")
    print(f"  отвергнуто:           {data.get('rejected_boxes')}")
    print(f"  перемещено изображений: {data.get('moved_to_rejected')}")
    for r, c in data.get("rejected_by_reason", {}).items():
        if c:
            print(f"    {r}: {c}")

# Финальные метрики
metrics_file = Path("runs/train/results.csv")
if metrics_file.exists():
    df = pd.read_csv(metrics_file)
    last = df.iloc[-1]
    print(f"\n[Обучение — последняя эпоха]")
    print(f"  epoch:        {int(last.get('epoch', -1))}")
    print(f"  mAP@0.5:      {last.get('metrics/mAP50(B)', 'n/a')}")
    print(f"  mAP@0.5:0.95: {last.get('metrics/mAP50-95(B)', 'n/a')}")
    print(f"  precision:    {last.get('metrics/precision(B)', 'n/a')}")
    print(f"  recall:       {last.get('metrics/recall(B)', 'n/a')}")

print("\n" + "=" * 60)